In [1]:

import yaml
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Protocol, runtime_checkable

import yaml

In [2]:
# --- In-memory schema representation ---

@dataclass(frozen=True)
class SchemaSpec:
    """Neutral, tool-agnostic schema representation produced from YAML."""
    name: str
    version: str | None
    raw: dict[str, Any]


# --- Schema definition layer (YAML -> in-memory schema) ---

@runtime_checkable
class SchemaLoader(Protocol):
    """Loads schema definition from YAML (or other source)."""
    def load(self, schema_ref: str) -> SchemaSpec: ...


class YamlFileSchemaLoader:
    """
    Loads schemas from YAML files.

    Examples:
      loader = YamlFileSchemaLoader(root_dir="schemas")
      spec = loader.load("user")              # -> schemas/user.yaml
      spec = loader.load("schemas/user.yaml") # direct path
    """

    def __init__(self, root_dir: str | Path = "schemas") -> None:
        self.root_dir = Path(root_dir)

    def load(self, schema_ref: str) -> SchemaSpec:
        path = self._resolve(schema_ref)
        data = self._read_yaml(path)

        # Expected structure like:
        # user:
        #   schema: {...}
        #   rules:  {...}
        if not isinstance(data, dict) or len(data) != 1:
            raise ValueError(
                f"Schema YAML must have exactly one top-level key (e.g., 'user'). Got: {list(data) if isinstance(data, dict) else type(data)}"
            )

        name, body = next(iter(data.items()))
        if not isinstance(name, str) or not isinstance(body, dict):
            raise ValueError("Top-level schema key must map to a dict.")

        schema_block = body.get("schema")
        rules_block = body.get("rules")
        version = body.get("version")  # optional

        if not isinstance(schema_block, dict):
            raise ValueError(f"Missing or invalid '{name}.schema' block (must be a dict).")
        if rules_block is not None and not isinstance(rules_block, dict):
            raise ValueError(f"Invalid '{name}.rules' block (must be a dict if present).")
        if version is not None and not isinstance(version, (str, int, float)):
            raise ValueError(f"Invalid '{name}.version' (must be scalar if present).")

        # Keep the raw structure flexible; downstream layers interpret it
        return SchemaSpec(
            name=name,
            version=str(version) if version is not None else None,
            raw=body,
        )

    def _resolve(self, schema_ref: str) -> Path:
        p = Path(schema_ref)

        # If user passed a path that exists, use it
        if p.exists() and p.is_file():
            return p

        # Otherwise interpret schema_ref as a schema name like "user"
        # and resolve to <root_dir>/<schema_ref>.yaml
        candidate = (self.root_dir / f"{schema_ref}.yaml").resolve()
        if not candidate.exists():
            raise FileNotFoundError(f"Schema file not found for ref '{schema_ref}'. Tried: {candidate}")
        return candidate

    def _read_yaml(self, path: Path) -> dict[str, Any]:
        try:
            text = path.read_text(encoding="utf-8")
        except OSError as e:
            raise OSError(f"Failed to read schema file: {path}") from e

        try:
            data = yaml.safe_load(text)
        except yaml.YAMLError as e:
            raise ValueError(f"Invalid YAML in schema file: {path}") from e

        if data is None:
            raise ValueError(f"Schema file is empty: {path}")
        if not isinstance(data, dict):
            raise ValueError(f"Schema YAML must parse to a dict at root. Got: {type(data)}")
        return data

In [3]:
# --- Example usage ---

if __name__ == "__main__":
    loader: SchemaLoader = YamlFileSchemaLoader(root_dir="../schemas")
    spec = loader.load("user")  # expects schemas/user.yaml
    print(spec.name)            # "user"
    print(spec.version)         # None unless you add "version:" under user:
    print(spec.raw.keys())      # dict_keys(['schema', 'rules'])
    print(spec.raw["schema"].keys())

user
None
dict_keys(['schema', 'rules'])
dict_keys(['id', 'first_name', 'last_name', 'email', 'created_at', 'is_active', 'country', 'system_x_id'])


In [4]:
spec

SchemaSpec(name='user', version=None, raw={'schema': {'id': {'type': 'integer', 'nullable': False, 'description': 'Unique user identifier (internal incremental id in generated data)'}, 'first_name': {'type': 'string', 'nullable': False, 'checks': {'str_length': {'min_value': 1, 'max_value': 100}}, 'description': 'User first name'}, 'last_name': {'type': 'string', 'nullable': False, 'checks': {'str_length': {'min_value': 1, 'max_value': 100}}, 'description': 'User last name'}, 'email': {'type': 'string', 'nullable': False, 'description': 'User email address'}, 'created_at': {'type': 'string', 'nullable': False, 'description': 'ISO-8601 datetime string when the record was created'}, 'is_active': {'type': 'boolean', 'nullable': False, 'description': 'Whether the user is active'}, 'country': {'type': 'string', 'nullable': False, 'description': 'ISO 3166-1 alpha-2 country code'}, 'system_x_id': {'type': 'string', 'nullable': False, 'description': 'External System X identifier (may contain l

## Define the plan type

In [5]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Type
from pydantic import BaseModel

@dataclass(frozen=True)
class PydanticPlan:
    model: Type[BaseModel]          # compiled model class
    name: str                       # schema name (e.g., "user")
    raw_spec: dict[str, Any]        # optional: keep for debugging

In [6]:
from typing import Any, Type
from pydantic import BaseModel, Field, create_model
import re

TYPE_MAP: dict[str, Any] = {
    "string": str,
    "integer": int,
    "float": float,
    "boolean": bool,
}

def _field_kwargs_from_yaml(field_def: dict[str, Any]) -> dict[str, Any]:
    """
    Translate a single YAML field definition into Pydantic Field constraints.
    Supports a small subset: nullable, str_length, regex.
    """
    checks = field_def.get("checks") or {}
    desc = field_def.get("description")

    kwargs: dict[str, Any] = {}
    if desc:
        kwargs["description"] = desc

    # string length
    if "str_length" in checks:
        sl = checks["str_length"] or {}
        # YAML uses min_value/max_value; Pydantic uses min_length/max_length
        if "min_value" in sl:
            kwargs["min_length"] = sl["min_value"]
        if "max_value" in sl:
            kwargs["max_length"] = sl["max_value"]

    # regex (if you decide to also read it from rules later)
    # kwargs["pattern"] = ...  # Pydantic v2 uses 'pattern' for Field
    return kwargs


class PydanticPlanCompiler:
    def compile(self, spec: "SchemaSpec") -> PydanticPlan:
        schema = spec.raw.get("schema")
        if not isinstance(schema, dict):
            raise ValueError(f"{spec.name}: missing schema block")

        fields: dict[str, tuple[Any, Any]] = {}

        for fname, fdef in schema.items():
            if not isinstance(fdef, dict):
                raise ValueError(f"{spec.name}.{fname}: field def must be dict")

            tname = fdef.get("type")
            py_type = TYPE_MAP.get(tname)
            if py_type is None:
                raise ValueError(f"{spec.name}.{fname}: unsupported type '{tname}'")

            nullable = bool(fdef.get("nullable", True))
            annotated_type = py_type | None if nullable else py_type

            # required vs optional
            default = None if nullable else ...  # ... means required
            field_kwargs = _field_kwargs_from_yaml(fdef)
            fields[fname] = (annotated_type, Field(default, **field_kwargs))

        Model = create_model(f"{spec.name.title()}Model", **fields)  # type: ignore[arg-type]

        return PydanticPlan(model=Model, name=spec.name, raw_spec=spec.raw)

## Using the plan

In [7]:
def validate_record(plan: PydanticPlan, record: dict[str, Any]) -> BaseModel:
    # returns validated object; raises pydantic.ValidationError if invalid
    return plan.model.model_validate(record)

In [8]:
# example
spec = loader.load("user")
plan = PydanticPlanCompiler().compile(spec)

In [9]:
user_obj = validate_record(plan, {"id": 11, "first_name": "Christopher", "last_name": "Williams", "email": "hernandezernest@example.net", "created_at": "1993-08-26T21:53:05.641380", "is_active": True, "country": "MV", "system_x_id": "400156"})

In [10]:
user_obj = validate_record(plan, {"id": 11, "first_name": "", "last_name": "Williams", "email": "hernandezernest@example.net", "created_at": "1993-08-26T21:53:05.641380", "is_active": True, "country": "MV", "system_x_id": "400156"})

ValidationError: 1 validation error for UserModel
first_name
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short